# Домашнее задание 3. Парсинг, Git и тестирование на Python

**Цели задания:**

* Освоить базовые подходы к web-scraping с библиотеками `requests` и `BeautisulSoup`: навигация по страницам, извлечение HTML-элементов, парсинг.
* Научиться автоматизировать задачи с использованием библиотеки `schedule`.
* Попрактиковаться в использовании Git и оформлении проектов на GitHub.
* Написать и запустить простые юнит-тесты с использованием `pytest`.


В этом домашнем задании вы разработаете систему для автоматического сбора данных о книгах с сайта [Books to Scrape](http://books.toscrape.com). Нужно реализовать функции для парсинга всех страниц сайта, извлечения информации о книгах, автоматического ежедневного запуска задачи и сохранения результата.

Важной частью задания станет оформление проекта: вы создадите репозиторий на GitHub, оформите `README.md`, добавите артефакты (код, данные, отчеты) и напишете базовые тесты на `pytest`.



In [ ]:
! pip install -q schedule pytest # установка библиотек, если ещё не

In [ ]:
# Библиотеки, которые могут вам понадобиться
# При необходимости расширяйте список
import time
import requests
import schedule
from bs4 import BeautifulSoup
import re
from pathlib import Path
from datetime import datetime
import pathlib


## Задание 1. Сбор данных об одной книге (20 баллов)

В этом задании мы начнем подготовку скрипта для парсинга информации о книгах со страниц каталога сайта [Books to Scrape](https://books.toscrape.com/).

Для начала реализуйте функцию `get_book_data`, которая будет получать данные о книге с одной страницы (например, с [этой](http://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html)). Соберите всю информацию, включая название, цену, рейтинг, количество в наличии, описание и дополнительные характеристики из таблицы Product Information. Результат достаточно вернуть в виде словаря.

**Не забывайте про соблюдение PEP-8** — помимо качественно написанного кода важно также документировать функции по стандарту:
* кратко описать, что она делает и для чего нужна;
* какие входные аргументы принимает, какого они типа и что означают по смыслу;
* аналогично описать возвращаемые значения.

*P. S. Состав, количество аргументов функции и тип возвращаемого значения можете менять как вам удобно. То, что написано ниже в шаблоне — лишь пример.*

In [ ]:

def get_book_data(book_url: str) -> dict:
    """
    Извлекает данные о книге с страницы интернет-магазина.

    Args:
        book_url (str): URL страницы книги (например,
                   http://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html)

    Returns:
        dict: Словарь с полями:
            - title (str): название книги
            - price (float): цена в фунтах (число)
            - rating (int): рейтинг (0–5)
            - in_stock (int): количество в наличии
            - description (str): описание
            - product_info (dict): дополнительные характеристики из таблицы Product Information
                (UPC, Product Type, Price (excl. tax), Price (incl. tax), Tax, Availability, Number of reviews)
    """

    # НАЧАЛО ВАШЕГО РЕШЕНИЯ
    try:
        # Выполняем GET-запрос
        response = requests.get(book_url, timeout=10)
        response.raise_for_status()  # Проверяем статус ответа

        # Парсим HTML
        soup = BeautifulSoup(response.content, 'html.parser')

        # Извлекаем название
        title = soup.find('h1').get_text(strip=True)

        # Извлекаем цену (в формате £12.99 → 12.99)
        price_text = soup.find('p', class_='price_color').get_text(strip=True)
        price = float(price_text.replace('£', ''))

        # Извлекаем рейтинг (класс типа 'star-rating Three')
        rating_elem = soup.find('p', class_='star-rating')
        rating_classes = rating_elem['class']
        rating_words = [c for c in rating_classes if c != 'star-rating']
        rating_word = rating_words[0] if rating_words else 'Zero'

        rating_map = {
            'Zero': 0,
            'One': 1,
            'Two': 2,
            'Three': 3,
            'Four': 4,
            'Five': 5
        }
        rating = rating_map.get(rating_word, 0)

        # Извлекаем количество в наличии (например, 'In stock (18 available)')
        stock_text = soup.find('p', class_='instock availability').get_text(strip=True)

        # Ищем число в скобках
        stock_match = re.search(r'\((\d+)\s*available\)', stock_text)
        in_stock = int(stock_match.group(1)) if stock_match else 0

        # Извлекаем описание (раздел после <h2>Product Description</h2>)
        description_elem = soup.find('div', id='product_description')
        if description_elem and description_elem.find_next_sibling('p'):
            description = description_elem.find_next_sibling('p').get_text(strip=True)
        else:
            description = ''

        # Извлекаем таблицу Product Information
        product_info = {}
        table = soup.find('table', class_='table table-striped')
        if table:
            for row in table.find_all('tr'):
                key = row.find('th').get_text(strip=True)
                value = row.find('td').get_text(strip=True)
                product_info[key] = value

        return {
            'title': title,
            'price': price,
            'rating': rating,
            'in_stock': in_stock,
            'description': description,
            'product_info': product_info
        }

    except requests.RequestException as e:
        print(f"Ошибка при запросе к {book_url}: {e}")
        return {}
    except Exception as e:
        print(f"Неожиданная ошибка при парсинге: {e}")
        return {}
    

    if __name__ == '__main__':
        book_url = "http://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html"
        data = get_book_data(book_url)
        print(data)
    # КОНЕЦ ВАШЕГО РЕШЕНИЯ

In [ ]:
# Используйте для самопроверки
book_url = 'http://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html'
get_book_data(book_url)

## Задание 2. Сбор данных обо всех книгах (20 баллов)

Создайте функцию `scrape_books`, которая будет проходиться по всем страницам из каталога (вида `http://books.toscrape.com/catalogue/page-{N}.html`) и осуществлять парсинг всех страниц в цикле, используя ранее написанную `get_book_data`.

Добавьте аргумент-флаг, который будет отвечать за сохранение результата в файл: если он будет равен `True`, то информация сохранится в ту же папку в файл `books_data.txt`; иначе шаг сохранения будет пропущен.

**Также не забывайте про соблюдение PEP-8**

In [ ]:
def scrape_books(start_page: int = 1, end_page: int = 50, save_to_file: bool = False) -> list:
    """
    Парсит книги со страниц каталога books.toscrape.com.

    Args:
        start_page (int): Номер начальной страницы (по умолчанию 1)
        end_page (int): Номер конечной страницы (по умолчанию 50)
        save_to_file (bool): Флаг сохранения результата в файл books_data.txt

    Returns:
        list: Список словарей с данными о книгах
    """

    # НАЧАЛО ВАШЕГО РЕШЕНИЯ
    all_books = []
    base_url = "http://books.toscrape.com/catalogue/page-{}.html"

    print(f"Начинаем парсинг страниц с {start_page} по {end_page}...")

    for page_num in range(start_page, end_page + 1):
        page_url = base_url.format(page_num)
        print(f"Обрабатываю страницу {page_num}: {page_url}")

        try:
            # Получаем HTML страницы каталога
            response = requests.get(page_url, timeout=10)
            response.raise_for_status()

            soup = BeautifulSoup(response.content, 'html.parser')

            # Находим все ссылки на книги на странице
            book_links = soup.find_all('h3')

            for book_link in book_links:
                # Извлекаем относительный URL книги
                relative_url = book_link.find('a')['href']
                # Формируем полный URL
                book_url = f"http://books.toscrape.com/catalogue/{relative_url}"

                # Парсим данные книги
                book_data = get_book_data(book_url)
                if book_data:  # Если данные успешно получены
                    all_books.append(book_data)
                    print(f"  ✓ Добавлена книга: {book_data['title']}")
                else:
                    print(f"  ✗ Не удалось получить данные для книги по URL: {book_url}")

                    # Небольшая задержка, чтобы не перегружать сервер
                    time.sleep(0.1)

        except requests.RequestException as e:
            print(f"Ошибка при загрузке страницы {page_num}: {e} ")
            continue
        except Exception as e:
            print(f"Неожиданная ошибка на странице {page_num}: {e}")
            continue

    print(f"\nПарсинг завершён. Всего собрано {len(all_books)} книг.")

    # Сохранение в файл, если флаг установлен
    if save_to_file:
        file_path = Path("books_data.txt")
        try:
            with open(file_path, "w", encoding="utf-8") as f:
                for i, book in enumerate(all_books, 1):
                    f.write(f"--- Книга {i} ---\n")
                    for key, value in book.items():
                        if key == "product_info":
                            f.write("product_info:\n")
                            for k, v in value.items():
                                f.write(f"  {k}: {v}\n")
                        else:
                            f.write(f"{key}: {value}\n")
                    f.write("\n")
            print(f"Данные сохранены в файл: {file_path.resolve()}")
        except IOError as e:
            print(f"Ошибка при сохранении файла: {e}")

    return all_books


# Пример использования
if __name__ == "__main__":
    # Парсим первые 3 страницы и сохраняем результат в файл
    books = scrape_books(start_page=1, end_page=3, save_to_file=True)
    # Или без сохранения в файл
    # books = scrape_books(start_page=1, end_page=2, save_to_file=False)
    # КОНЕЦ ВАШЕГО РЕШЕНИЯ

In [ ]:
# Проверка работоспособности функции
res = scrape_books(is_save=True) # Допишите ваши аргументы
print(type(res), len(res)) # и проверки

## Задание 3. Настройка регулярной выгрузки (10 баллов)

Настройте автоматический запуск функции сбора данных каждый день в 19:00.
Для автоматизации используйте библиотеку `schedule`. Функция должна запускаться в указанное время и сохранять обновленные данные в текстовый файл.



Бесконечный цикл должен обеспечивать постоянное ожидание времени для запуска задачи и выполнять ее по расписанию. Однако чтобы не перегружать систему, стоит подумать о том, чтобы выполнять проверку нужного времени не постоянно, а раз в какой-то промежуток. В этом вам может помочь `time.sleep(...)`.

Проверьте работоспособность кода локально на любом времени чч:мм.



In [ ]:
# НАЧАЛО ВАШЕГО РЕШЕНИЯ
def daily_scrape():
    """Функция для ежедневного запуска парсинга и сохранения данных."""
    print(f"[{datetime.now()}] Запуск ежедневного парсинга...")

    # Парсим данные
    books = scrape_books(start_page=1, end_page=3)  # Для примера берём 3 страницы

    # Формируем имя файла с датой
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"books_data_{timestamp}.txt"
    filepath = pathlib.Path(filename)

    # Сохраняем в файл
    try:
        with open(filepath, "w", encoding="utf-8") as f:
            f.write(f"Данные собраны: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
            f.write(f"Всего книг: {len(books)}\n\n")

            for i, book in enumerate(books, 1):
                f.write(f"--- Книга {i} ---\n")
                for key, value in book.items():
                    if key == "product_info":
                        f.write("product_info:\n")
                        for k, v in value.items():
                            f.write(f"{k}: {v}\n")
                    else:
                        f.write(f"{key}: {value}\n")
                        f.write("\n")
                        print(f"[{datetime.now()}] Данные сохранены в {filepath}")
    except IOError as e:
        print(f"[{datetime.now()}] Ошибка при сохранении файла: {e}")


def main():
    """Основная функция с планировщиком."""
    # Настраиваем расписание
    schedule.every().day.at("19:00").do(daily_scrape)
    # Для тестирования можно добавить запуск в ближайшее время:
    # schedule.every().minutes.do(daily_scrape)  # Каждые минуты (для теста)

    print("Планировщик запущен. Ожидание времени запуска...")
    print("Для выхода Ctrl+C")

    # Бесконечный цикл с проверкой расписания
    while True:
        schedule.run_pending()  # Проверяем, есть ли запланированные задачи
        time.sleep(30)  # Проверяем каждые 30 секунд (оптимальный интервал)


if __name__ == "__main__":
    main()
# КОНЕЦ ВАШЕГО РЕШЕНИЯ

## Задание 4. Написание автотестов (15 баллов)

Создайте минимум три автотеста для ключевых функций парсинга — например, `get_book_data` и `scrape_books`. Идеи проверок (можете использовать свои):

* данные о книге возвращаются в виде словаря с нужными ключами;
* список ссылок или количество собранных книг соответствует ожиданиям;
* значения отдельных полей (например, `title`) корректны.

Оформите тесты в отдельном скрипте `tests/test_scraper.py`, используйте библиотеку `pytest`. Убедитесь, что тесты проходят успешно при запуске из терминала командой `pytest`.

Также выведите результат их выполнения в ячейке ниже.

**Не забывайте про соблюдение PEP-8**


In [ ]:
# Ячейка для демонстрации работоспособности
# Сам код напишите в отдельном скрипте
! pytest test/test_scraper.py

## Задание 5. Оформление проекта на GitHub и работа с Git (35 баллов)

В этом задании нужно воспользоваться системой контроля версий Git и платформой GitHub для хранения и управления своим проектом. **Ссылку на свой репозиторий пришлите в форме для сдачи ответа.**

### Пошаговая инструкция и задания

**1. Установите Git на свой компьютер.**

* Для Windows: [скачайте установщик](https://git-scm.com/downloads) и выполните установку.
* Для macOS:

  ```
  brew install git
  ```
* Для Linux:

  ```
  sudo apt update
  sudo apt install git
  ```

**2. Настройте имя пользователя и email.**

Это нужно для подписи ваших коммитов, сделайте в терминале через `git config ...`.

**3. Создайте аккаунт на GitHub**, если у вас его еще нет:
[https://github.com](https://github.com)

**4. Создайте новый репозиторий на GitHub:**

* Найдите кнопку **New repository**.
* Укажите название, краткое описание, выберите тип **Public** (чтобы мы могли проверить ДЗ).
* Не ставьте галочку Initialize this repository with a README.

**5. Создайте локальную папку с проектом.** Можно в терминале, можно через UI, это не имеет значения.

**6. Инициализируйте Git в этой папке.** Здесь уже придется воспользоваться некоторой командой в терминале.

**7. Привяжите локальный репозиторий к удаленному на GitHub.**

**8. Создайте ветку разработки.** По умолчанию вы будете находиться в ветке `main`, создайте и переключитесь на ветку `hw-books-parser`.

**9. Добавьте в проект следующие файлы и папки:**

* `scraper.py` — ваш основной скрипт для сбора данных.
* `README.md` — файл с кратким описанием проекта:

  * цель;
  * инструкции по запуску;
  * список используемых библиотек.
* `requirements.txt` — файл со списком зависимостей, необходимых для проекта (не присылайте все из глобального окружения, создайте изолированную виртуальную среду, добавьте в нее все нужное для проекта и получите список библиотек через `pip freeze`).
* `artifacts/` — папка с результатами парсинга (`books_data.txt` — полностью или его часть, если весь не поместится на GitHub).
* `notebooks/` — папка с заполненным ноутбуком `HW_03_python_ds_2025.ipynb` и запущенными ячейками с выводами на экран.
* `tests/` — папка с тестами на `pytest`, оформите их в формате скрипта(-ов) с расширением `.py`.
* `.gitignore` — стандартный файл, который позволит исключить временные файлы при добавлении в отслеживаемые (например, `__pycache__/`, `.DS_Store`, `*.pyc`, `venv/` и др.).


**10. Сделайте коммит.**

**11. Отправьте свою ветку на GitHub.**

**12. Создайте Pull Request:**

* Перейдите в репозиторий на GitHub.
* Нажмите кнопку **Compare & pull request**.
* Укажите, что было добавлено, и нажмите **Create pull request**.

**13. Выполните слияние Pull Request:**

* Убедитесь, что нет конфликтов.
* Нажмите **Merge pull request**, затем **Confirm merge**.

**14. Скачайте изменения из основной ветки локально.**



### Требования к итоговому репозиторию

* Файл `scraper.py` с рабочим кодом парсера.
* `README.md` с описанием проекта и инструкцией по запуску.
* Папка `artifacts/` с результатом сбора данных (`.txt` файл).
* Папка `tests/` с тестами на `pytest`.
* Папка `notebooks/` с заполненным ноутбуком `HW_03_python_ds_2025.ipynb`.
* Pull Request с комментарием из ветки `hw-books-parser` в ветку `main`.
* Примерная структура:

  ```
  books_scraper/
  ├── artifacts/
  │   └── books_data.txt
  ├── notebooks/
  │   └── HW_03_python_ds_2025.ipynb
  ├── scraper.py
  ├── README.md
  ├── tests/
  │   └── test_scraper.py
  ├── .gitignore
  └── requirements.txt
  ```